# Step 7 (Kaggle GPU) — Finite-temperature MD: ferroelectric transition of BaTiO3 and PbTiO3
NPT molecular dynamics (4×4×4 supercell, 320 atoms) with stepwise heating, using the best models of the static benchmark.
Order parameter: average off-centring of Ti relative to its O6 cage (Å) and lattice parameters a, b, c.

**Run on Kaggle:** Settings → Accelerator **GPU T4 x2** (or P100), Internet **On**. Set `FAMILY` in cell 1 (`"orb"`, then `"mace"` in a second notebook). Save Version → Save & Run All (Commit). Output: `md_<family>.csv` (written after every temperature).

In [ ]:
FAMILY = "orb"     # run twice: "orb" (ORB-v3 MPA), then "mace" (MACE-MPA-0)
PKG = {"chgnet": "chgnet", "sevennet": "sevenn", "mattersim": "mattersim", "orb": "orb-models"}[FAMILY]
if FAMILY == "orb":   # orb-models pins an old dm-tree without Python 3.13 wheels
    !pip install -q "dm-tree>=0.1.9" cached_path phonopy spglib "ase>=3.23"
    !pip install -q --no-deps "orb-models>=0.5,<0.6"
    import importlib.metadata as _md, subprocess as _sp
    _reqs = [r.split(";")[0].strip() for r in (_md.requires("orb-models") or [])
             if "extra ==" not in r and not r.lower().replace("_", "-").startswith("dm-tree")]
    print("orb-models", _md.version("orb-models"), "| installing dependencies:", _reqs)
    _sp.run(["pip", "install", "-q", *_reqs])
else:
    !pip install -q {PKG} phonopy spglib "ase>=3.23"


In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import numpy as np, pandas as pd, torch, time, importlib.metadata as md
from ase import Atoms, units
from ase.constraints import FixSymmetry
from ase.filters import FrechetCellFilter
from ase.optimize import BFGS
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.md.nptberendsen import Inhomogeneous_NPTBerendsen
from ase.neighborlist import neighbor_list
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "|", PKG, md.version(PKG))
MODEL = {"sevennet": "7net-mf-ompa", "orb": "orb-v3-conservative-inf-mpa", "mattersim": "MatterSim-v1.0.0-1M",
         "mace": "medium-mpa-0", "chgnet": "chgnet-0.3.0"}[FAMILY]

def get_calc(name):
    if FAMILY == "mace":
        from mace.calculators import mace_mp
        return mace_mp(model=name, device=device, default_dtype="float32")
    if FAMILY == "chgnet":
        from chgnet.model import CHGNet
        from chgnet.model.dynamics import CHGNetCalculator
        return CHGNetCalculator(CHGNet.load(model_name=name.split("-")[1]), use_device=device)
    if FAMILY == "sevennet":
        from sevenn.calculator import SevenNetCalculator
        if name == "7net-mf-ompa":
            return SevenNetCalculator(model=name, modal="mpa", device=device)
        return SevenNetCalculator(model=name, device=device)
    if FAMILY == "mattersim":
        from mattersim.forcefield import MatterSimCalculator
        return MatterSimCalculator(load_path=name + ".pth", device=device)
    if FAMILY == "orb":
        import pkgutil, importlib, inspect, orb_models
        from orb_models.forcefield import pretrained
        # find ORBCalculator wherever this version of orb-models puts it
        Calc = None
        for mi in pkgutil.walk_packages(orb_models.__path__, "orb_models."):
            if "calculator" not in mi.name: continue
            try: mod = importlib.import_module(mi.name)
            except Exception as ex: print("cannot import", mi.name, repr(ex)[:200]); continue
            if hasattr(mod, "ORBCalculator"): Calc = mod.ORBCalculator; print("ORBCalculator from", mi.name); break
        assert Calc is not None, "ORBCalculator not found"
        fname = name.replace("-", "_")
        avail = [f for f in dir(pretrained) if f.startswith("orb")]
        print("available ORB models:", avail)
        assert fname in avail, f"{fname} not in this orb-models version"
        fn = getattr(pretrained, fname)
        kw = {"device": device}
        if "precision" in inspect.signature(fn).parameters: kw["precision"] = "float32-highest"
        orbff = fn(**kw)
        ckw = {"device": device} if "device" in inspect.signature(Calc).parameters else {}
        return Calc(orbff, **ckw)
SYSTEMS = {"BaTiO3": ("Ba","Ti"), "PbTiO3": ("Pb","Ti"), "KNbO3": ("K","Nb"),
           "SrTiO3": ("Sr","Ti"), "KTaO3": ("K","Ta")}
DIRS = {"Pm-3m": None, "P4mm": [0,0,1], "Amm2": [1,1,0], "R3m": [1,1,1]}
QP = {"G": [0,0,0], "X": [0,0,.5], "M": [.5,.5,0], "R": [.5,.5,.5]}
THz2cm = 33.35641

calc = get_calc(MODEL)
OUT = f"/kaggle/working/md_{FAMILY}.csv"
SETUP = {"PbTiO3": None, "BaTiO3": dict(A="Ba", B="Ti", gs=[1, 1, 1], T=list(range(50, 601, 50))),
}
SETUP["PbTiO3"] = dict(A="Pb", B="Ti", gs=[0, 0, 1], T=list(range(200, 1501, 100)))
N, DT, NSTEP, NAVG = 4, 2.0, 2500, 1250   # 4x4x4 supercell (320 atoms), 5 ps per temperature
T_START = time.time(); BUDGET = 11.0 * 3600   # stop cleanly before the 12 h Kaggle limit

def unit(A, B, d, a=4.0):
    frac = [[0,0,0],[.5,.5,.5],[.5,.5,0],[.5,0,.5],[0,.5,.5]]
    at = Atoms([A, B, "O", "O", "O"], scaled_positions=frac, cell=np.eye(3)*a, pbc=True)
    d = np.array(d, float); d /= np.linalg.norm(d); p = at.get_positions(); p[1] += 0.1*d; p[2:] -= 0.06*d; at.set_positions(p)
    return at

def relax(at):
    at = at.copy(); at.calc = calc; at.set_constraint(FixSymmetry(at, symprec=1e-4))
    BFGS(FrechetCellFilter(at), logfile=None).run(fmax=1e-3, steps=2000); at.set_constraint(); return at

def cages(at, B):
    i, j = neighbor_list("ij", at, 2.2)
    sym = np.array(at.get_chemical_symbols())
    idx = [k for k in range(len(at)) if sym[k] == B]
    cage = {b: [jj for ii, jj in zip(i, j) if ii == b and sym[jj] == "O"] for b in idx}
    assert all(len(v) == 6 for v in cage.values()), "O6 cage detection failed"
    return idx, cage

def order_param(at, idx, cage):
    pos = at.get_positions(); cell = at.cell; d = []
    for b in idx:
        v = pos[cage[b]] - pos[b]
        v = v - np.round(v @ np.linalg.inv(cell[:])) @ cell[:]
        d.append(-v.mean(0))           # B displacement relative to cage centre
    d = np.array(d)
    return d.mean(0), np.linalg.norm(d, axis=1).mean()

done = pd.read_csv(OUT) if os.path.exists(OUT) else pd.DataFrame()
MATERIALS = ["PbTiO3", "BaTiO3"]
if FAMILY == "orb": SETUP["BaTiO3"]["T"] = list(range(650, 1001, 50))   # 50-600 K already done with ORB
for mat, s in SETUP.items():
    if mat not in MATERIALS: continue
    uc = relax(unit(s["A"], s["B"], s["gs"]))
    at = uc.repeat((N, N, N)); at.calc = calc
    ref = Atoms(uc.get_chemical_symbols(), scaled_positions=[[0,0,0],[.5,.5,.5],[.5,.5,0],[.5,0,.5],[0,.5,.5]],
                cell=np.eye(3)*4.0, pbc=True).repeat((N, N, N))     # same atom order, undistorted
    idx, cage = cages(ref, s["B"])
    MaxwellBoltzmannDistribution(at, temperature_K=s["T"][0]); Stationary(at)
    for T in s["T"]:
        if time.time() - T_START > BUDGET: print("time budget reached, stopping"); break
        if len(done) and ((done.material == mat) & (done.T_target == T)).any(): continue
        t0 = time.time()
        dyn = Inhomogeneous_NPTBerendsen(at, timestep=DT*units.fs, temperature_K=T, pressure_au=0.0,
                                         taut=100*units.fs, taup=1000*units.fs, compressibility_au=4.57e-5/units.bar)
        rec = []
        def sample():
            if dyn.nsteps >= NSTEP - NAVG and dyn.nsteps % 10 == 0:
                P, mag = order_param(at, idx, cage); cp = at.cell.cellpar()
                rec.append([*P, np.linalg.norm(P), mag, cp[0]/N, cp[1]/N, cp[2]/N, at.get_temperature()])
        dyn.attach(sample, interval=1); dyn.run(NSTEP)
        r = np.array(rec).mean(0); abc = sorted(r[5:8])
        row = dict(model=MODEL, material=mat, T_target=T, T_md=r[8], Px=r[0], Py=r[1], Pz=r[2], P_abs=r[3], local_disp=r[4],
                   a=r[5], b=r[6], c=r[7], tetragonality=abc[2]/abc[0], time_s=round(time.time()-t0))
        done = pd.concat([done, pd.DataFrame([row])], ignore_index=True); done.to_csv(OUT, index=False)
        print(f"   ({(time.time()-t0)/NSTEP:.3f} s/step)")
        print(f"{MODEL} {mat} T={T:5d} K  |<d>|={r[3]:.3f} A  <|d|>={r[4]:.3f} A  c/a~{abc[2]/abc[0]:.4f}  {time.time()-t0:.0f}s", flush=True)
print("done ->", OUT)
